# 09 · WGCNA: gene modules in your RNA-seq data (RGM) and in the JDM lab, one network per study

**Run order:** after 02 to 04 (RGM) and with `4b_rpkm.csv` on `/mnt/file-systems/`. Kernel: **R (endotypes)**, with the `WGCNA` package installed. This notebook runs for a while (10 to 30 minutes per network is normal).

## What WGCNA does, in your own data's terms
So far every notebook compared genes one at a time or whole samples. WGCNA looks at **groups of genes that rise and fall together across samples**. Genes that move together form a **module** (given an arbitrary colour name such as `turquoise`). Each module is summarised by one number per sample, its **eigengene** (ME), which says how strongly that sample expresses the module. A module can then be tested against traits: does the ME differ between CANDLE and healthy? Is it high where the interferon score is high? One module is expected to hold the interferon response genes. Anne finds one in each of her studies (`black`, `lightgreen`, `magenta`).

## Why one network per study (Anne's notebooks 13 to 15, and the same rule as everywhere else)
Your RGM samples and the JDM-lab samples come from different labs and different pipelines (counts vs RPKM). A network built on the two pooled would mostly find the lab difference. So there are **two networks**, and the two are then compared with Anne's **module preservation** test: do genes that form a module in one study also hang together in the other?

## Settings (Anne's, from notebooks 13 and 14)
Signed network and TOM; power = the lowest power with scale-free R-squared at least 0.90 where the curve has flattened, else the WGCNA FAQ value for the sample size (signed: 18, 16, 14, 12 for under 20, 20 to 30, 30 to 40, over 40 samples); `minModuleSize` 30; `mergeCutHeight` 0.25; `maxBlockSize` 5000.

## The two inputs
- **RGM:** the **ComBat-seq counts of notebook 04**, one sample per patient (facilities already corrected, diagnosis protected). Genes with fewer than 10 counts in more than 90% of samples are removed (WGCNA FAQ), then **PFlog1pPF** normalisation (Booeshaghi et al. 2022), exactly as Anne does for her counts. Diagnoses with fewer than 3 samples are left out.
- **JDM lab:** log2(RPKM + 1), quantile-normalised (that lab's own recipe, as in 07). RPKM is not counts, so Anne's count filter becomes: genes with RPKM below 1 in more than 90% of samples are removed.

**Reads:** `rnaseq_out_v2/batch_correction/counts_combatseq_one_per_patient.csv` and `rnaseq_samples_eligible.csv` (03 and 04); `/mnt/file-systems/4b_rpkm.csv`; `genes/ifn-response-genes-28.txt`, `genes/nfkb-only-control-11-dejesus2020.txt`.
**Writes (all CSV, plots inline):** `rnaseq_out_v2/wgcna/RGM/` and `.../JDM/` each with `module_genes.csv`, `module_summary.csv`, `module_trait.csv`, `ifn_module.csv`, `power_fit.csv`; and in `wgcna/`: `preservation_RGM_modules_in_JDM.csv`, `preservation_JDM_modules_in_RGM.csv`, `module_overlap_RGM_JDM.csv`, `PROCESSING_LOG_wgcna.md`.

---
# Part 1 · Settings
| variable | default | meaning |
|---|---|---|
| `MIN_PER_GROUP` | `3` | a diagnosis with fewer samples than this is left out of the RGM network |
| `RPKM_MIN` | `1` | JDM filter: genes below this RPKM in more than 90% of samples are removed |
| `N_PERM` | `200` | random gene sets used for the preservation test (Anne uses 1,000; 200 keeps the run time down) |
| `MIN_OVERLAP` | `10` | a pair of modules is listed in the overlap table only if they share at least this many genes |
| `DROP_SAME_LIBRARY` | `TRUE` | drop JDM-lab samples that are the same library as an RGM sample (as in 04b and 07) |

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
BC_DIR  <- file.path(IN_DIR, "batch_correction")
WG_DIR  <- file.path(IN_DIR, "wgcna")
dir.create(WG_DIR, showWarnings = FALSE, recursive = TRUE)
MNT      <- "/mnt/file-systems/"
JDM_RPKM <- paste0(MNT, "4b_rpkm.csv")
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"

MIN_PER_GROUP <- 3
RPKM_MIN      <- 1
N_PERM        <- 200
MIN_OVERLAP   <- 10
DROP_SAME_LIBRARY <- TRUE
HC_PATTERN  <- "^(hc|relative-hc|healthy|control)"
JDM_PATTERN <- "jdm|dermatomyositis"
RECODE <- c("Undifferentiated Interferonopathy" = "UAID")
SEED <- 20260928

need_files <- c(file.path(BC_DIR, "counts_combatseq_one_per_patient.csv"),
                file.path(IN_DIR, "rnaseq_samples_eligible.csv"), JDM_RPKM)
absent <- need_files[!file.exists(need_files)]
if (length(absent)) stop("Missing file(s):\n  ", paste(absent, collapse = "\n  "), "\nNotebooks 02, 03 and 04 must have run first.")
if (!requireNamespace("WGCNA", quietly = TRUE))
  stop("The WGCNA package is not installed. In a terminal: conda install -y -p /home/jovyan/envs/endotypes -c conda-forge r-wgcna  (then restart the kernel).")
suppressMessages({library(WGCNA); library(limma)})
options(stringsAsFactors = FALSE)
set.seed(SEED)

read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p, warn = FALSE)); x[x != "" & !startsWith(x, "#")] }
irg <- read_set("ifn-response-genes-28.txt"); nfk <- read_set("nfkb-only-control-11-dejesus2020.txt")
by_symbol <- function(M, sym) {            # one row per gene symbol: the best-expressed copy
  o <- order(rowMeans(M, na.rm = TRUE), decreasing = TRUE)
  o <- o[!is.na(sym[o]) & sym[o] != "" & !duplicated(sym[o])]
  M <- M[o, , drop = FALSE]; rownames(M) <- sym[o]; M }
cat("settings OK; WGCNA", as.character(packageVersion("WGCNA")), "; output folder:", WG_DIR, "\n")

---
# Part 2 · The RGM input
ComBat-seq counts (notebook 04), one sample per patient. Genes with fewer than 10 counts in more than 90% of samples are removed; one row is kept per gene symbol (the best-expressed); then **PFlog1pPF**: scale each sample to the mean library size, log(1 + x), scale again (Booeshaghi AS et al. bioRxiv 2022, doi:10.1101/2022.05.06.490859; Anne's notebook 12).

In [ ]:
cs <- read.csv(file.path(BC_DIR, "counts_combatseq_one_per_patient.csv"), check.names = FALSE)
sym <- cs$Gene_Name
M <- as.matrix(cs[, !(colnames(cs) %in% c("Gene_ID", "Gene_Name"))]); storage.mode(M) <- "numeric"
s_all <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
stopifnot(!anyNA(match(colnames(M), s_all$sample_id)))
s <- s_all[match(colnames(M), s_all$sample_id), ]
dx <- s$dx_clean; dx[dx == ""] <- "(blank)"; hit <- dx %in% names(RECODE); dx[hit] <- unname(RECODE[dx[hit]]); s$diagnosis <- dx

n_dx <- table(s$diagnosis); small <- names(n_dx)[n_dx < MIN_PER_GROUP]
if (length(small)) cat("left out (fewer than", MIN_PER_GROUP, "samples):", paste0(small, " (n=", n_dx[small], ")"), "\n")
k <- !(s$diagnosis %in% small); M <- M[, k, drop = FALSE]; s <- s[k, ]

keep_g <- rowMeans(M < 10) <= 0.90
M <- M[keep_g, , drop = FALSE]; sym <- sym[keep_g]
M <- by_symbol(M, sym)
pf <- function(X) t(t(X) / colSums(X) * mean(colSums(X)))
E_rgm <- pf(log1p(pf(M)))
md_rgm <- data.frame(sample = colnames(E_rgm), diagnosis = s$diagnosis, batch = s$batch, stringsAsFactors = FALSE)
print(table(diagnosis = md_rgm$diagnosis))
c(RGM_samples = ncol(E_rgm), genes_after_filter = nrow(E_rgm))

---
# Part 3 · The JDM-lab input
`4b_rpkm.csv` (rows `Cohort` and `ID`, then genes by symbol). JDM or healthy is read from the `Cohort` row, as in 04b. Genes with RPKM below `RPKM_MIN` in more than 90% of samples are removed, the rest become log2(RPKM + 1) and are quantile-normalised across the JDM-lab samples. JDM-lab samples that are the same library as an RGM sample are dropped (`DROP_SAME_LIBRARY`).

In [ ]:
raw <- read.csv(JDM_RPKM, check.names = FALSE, colClasses = "character")
gcol <- colnames(raw)[1]
is_meta <- raw[[gcol]] %in% c("Cohort", "ID")
ids <- unlist(raw[raw[[gcol]] == "ID", -1], use.names = FALSE)
cohort <- if ("Cohort" %in% raw[[gcol]]) unlist(raw[raw[[gcol]] == "Cohort", -1], use.names = FALSE) else rep("", length(ids))
cohort[is.na(cohort)] <- ""
R <- as.matrix(raw[!is_meta, -1]); storage.mode(R) <- "numeric"
rownames(R) <- raw[[gcol]][!is_meta]; colnames(R) <- ids

clean_label <- function(x) trimws(sub("\\.\\.\\.[0-9]+$", "", x))
classify <- function(x) ifelse(grepl(HC_PATTERN, x, ignore.case = TRUE), "HC", ifelse(grepl(JDM_PATTERN, x, ignore.case = TRUE), "JDM", NA))
grp <- classify(clean_label(cohort))
if (!any(grp == "HC", na.rm = TRUE)) grp <- classify(clean_label(ids))
if (!any(grp == "HC", na.rm = TRUE) || !any(grp == "JDM", na.rm = TRUE))
  stop("Could not find both healthy controls and JDM in 4b_rpkm.csv. Change HC_PATTERN / JDM_PATTERN in Part 1 (see notebook 04b for the label table).")
same_lib <- if (DROP_SAME_LIBRARY) vapply(ids, function(i) any(endsWith(s_all$sample_id, paste0("-", i))), logical(1)) else rep(FALSE, length(ids))
keep_j <- !is.na(grp) & !same_lib
cat("JDM-lab samples read:", length(ids), "| neither JDM nor HC:", sum(is.na(grp)), "| same library as an RGM sample:", sum(same_lib & !is.na(grp)), "| kept:", sum(keep_j), "\n")
R <- R[, keep_j, drop = FALSE]; md_jdm <- data.frame(sample = colnames(R), diagnosis = grp[keep_j], stringsAsFactors = FALSE)

low <- rowMeans(R < RPKM_MIN, na.rm = TRUE) > 0.90
R <- R[!low, , drop = FALSE]
E <- log2(R + 1)
full <- rowSums(is.na(E)) == 0; E[full, ] <- normalizeQuantiles(E[full, , drop = FALSE])
E_jdm <- by_symbol(E, rownames(E))
E_jdm <- E_jdm[rowSums(is.na(E_jdm)) == 0, , drop = FALSE]      # WGCNA needs a value for every sample
print(table(diagnosis = md_jdm$diagnosis))
c(JDM_lab_samples = ncol(E_jdm), genes_after_filter = nrow(E_jdm))

---
# Part 4 · One function builds a network and its summaries
`run_net()` is used for both studies, so they are treated identically. Per study it:
1. removes genes and samples WGCNA flags as unusable (`goodSamplesGenes`);
2. draws the **sample tree** (outliers show as a long lone branch);
3. chooses the **power** (table and plot);
4. finds the **modules** (`blockwiseModules`) and each gene's **kME** (its correlation with its module's eigengene);
5. tests every module's eigengene against **each diagnosis** (correlation with a 0/1 indicator), the **interferon score** (mean z-score of the 28 genes), and, for RGM, the **sequencing batch**;
6. finds the **interferon module**: the module holding most of the 28 interferon genes, with the 11 NF-kB-only control genes as the check (they should **not** be in it);
7. writes everything as CSV.

**Module-trait table.** Correlations are Pearson; p-values from `corPvalueStudent`; adjusted p by Benjamini-Hochberg over every module and trait. A module whose strongest link is to the sequencing batch and not to a diagnosis is flagged `batch?` in the summary: it may be technical.

In [ ]:
faq_power <- function(n) if (n < 20) 18 else if (n < 30) 16 else if (n < 40) 14 else 12

run_net <- function(tag, E, md, has_batch = FALSE) {
  out <- file.path(WG_DIR, tag); dir.create(out, showWarnings = FALSE, recursive = TRUE)
  datExpr <- t(E)
  gsg <- goodSamplesGenes(datExpr, verbose = 0)
  cat(tag, ": genes flagged", sum(!gsg$goodGenes), "| samples flagged", sum(!gsg$goodSamples), "\n")
  datExpr <- datExpr[gsg$goodSamples, gsg$goodGenes, drop = FALSE]; md <- md[gsg$goodSamples, ]
  n <- nrow(datExpr)

  options(repr.plot.width = 11, repr.plot.height = 4.5)
  plot(hclust(dist(datExpr), method = "average"), labels = FALSE, hang = -1, main = paste(tag, ": sample clustering (look for a lone outlier branch)"), xlab = "", sub = "")

  powers <- c(1:10, seq(12, 30, 2))
  invisible(capture.output(sft <- pickSoftThreshold(datExpr, powerVector = powers, networkType = "signed", verbose = 0)))
  fit <- data.frame(power = sft$fitIndices$Power,
                    scale_free_R2 = round(-sign(sft$fitIndices$slope) * sft$fitIndices$SFT.R.sq, 3),
                    mean_connectivity = round(sft$fitIndices$mean.k., 1))
  flat <- fit$power[fit$scale_free_R2 >= 0.90 & c(FALSE, abs(diff(fit$scale_free_R2)) < 0.01)]
  POWER <- if (length(flat)) min(flat) else faq_power(n)
  rule <- if (length(flat)) "lowest power with R2 >= 0.90 where the curve has flattened" else sprintf("WGCNA FAQ value for a signed network with %d samples", n)
  print(fit); cat("power:", POWER, "(", rule, ")\n")
  write.csv(fit, file.path(out, "power_fit.csv"), row.names = FALSE)
  par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
  plot(fit$power, fit$scale_free_R2, type = "b", xlab = "power", ylab = "scale-free fit R2", main = paste(tag, ": power")); abline(h = 0.90, col = "red", lty = 2)
  plot(fit$power, fit$mean_connectivity, type = "b", xlab = "power", ylab = "mean connectivity", main = "mean connectivity")

  net <- blockwiseModules(datExpr, power = POWER, networkType = "signed", TOMType = "signed",
                          minModuleSize = 30, mergeCutHeight = 0.25, maxBlockSize = 5000,
                          numericLabels = FALSE, pamRespectsDendro = FALSE, verbose = 0)
  colors <- net$colors
  sizes <- sort(table(colors), decreasing = TRUE)
  cat(tag, ":", sum(names(sizes) != "grey"), "modules holding", sum(sizes[names(sizes) != "grey"]), "of", length(colors), "genes;",
      unname(sizes["grey"]), "grey\n")
  print(sizes)
  ME <- orderMEs(net$MEs); ME <- ME[, colnames(ME) != "MEgrey", drop = FALSE]
  kME <- cor(datExpr, ME)
  modules <- sub("^ME", "", colnames(ME))
  in_mod <- names(colors)[colors != "grey"]
  module_genes <- data.frame(gene = in_mod, module = colors[in_mod], kME = kME[cbind(in_mod, paste0("ME", colors[in_mod]))], row.names = NULL)
  write.csv(module_genes, file.path(out, "module_genes.csv"), row.names = FALSE)

  ## traits
  dxs <- sort(unique(md$diagnosis))
  Tm <- sapply(dxs, function(d) as.numeric(md$diagnosis == d)); colnames(Tm) <- paste0("dx:", dxs)
  gi <- intersect(irg, colnames(datExpr))
  if (length(gi) >= 3) { Tm <- cbind(Tm, IFN_score = rowMeans(scale(datExpr[, gi, drop = FALSE]))) }
  if (has_batch) { bs <- sort(unique(md$batch)); Tb <- sapply(bs, function(b) as.numeric(md$batch == b)); colnames(Tb) <- paste0("batch:", bs); Tm <- cbind(Tm, Tb) }
  Tm <- Tm[, apply(Tm, 2, sd) > 0, drop = FALSE]
  r <- cor(ME, Tm, use = "pairwise.complete.obs"); p <- corPvalueStudent(r, n)
  mt <- data.frame(module = rep(modules, ncol(Tm)), trait = rep(colnames(Tm), each = length(modules)),
                   r = round(as.vector(r), 3), p = signif(as.vector(p), 3), stringsAsFactors = FALSE)
  mt$adj_p <- signif(p.adjust(mt$p, method = "BH"), 3)
  write.csv(mt, file.path(out, "module_trait.csv"), row.names = FALSE)

  ## module summary
  hubs <- sapply(modules, function(mo) { g <- names(colors)[colors == mo]; paste(head(names(sort(kME[g, paste0("ME", mo)], decreasing = TRUE)), 6), collapse = ", ") })
  dxcols <- grep("^dx:", colnames(r)); bcols <- grep("^batch:", colnames(r))
  best_dx <- apply(abs(r[, dxcols, drop = FALSE]), 1, which.max)
  summ <- data.frame(module = modules, genes = as.integer(sizes[modules]),
                     ifn_genes_in_module = vapply(modules, function(mo) sum(irg %in% names(colors)[colors == mo]), 1L),
                     strongest_diagnosis = colnames(r)[dxcols][best_dx],
                     r_with_it = round(r[cbind(seq_along(modules), dxcols[best_dx])], 2),
                     adj_p = mt$adj_p[match(paste(modules, colnames(r)[dxcols][best_dx]), paste(mt$module, mt$trait))],
                     max_abs_r_batch = if (length(bcols)) round(apply(abs(r[, bcols, drop = FALSE]), 1, max), 2) else NA_real_,
                     top_hub_genes = hubs, row.names = NULL, stringsAsFactors = FALSE)
  summ$flag <- ifelse(!is.na(summ$max_abs_r_batch) & summ$max_abs_r_batch > abs(summ$r_with_it), "batch?", "")
  summ <- summ[order(-summ$genes), ]
  write.csv(summ, file.path(out, "module_summary.csv"), row.names = FALSE)

  ## interferon module
  ifn <- NULL
  if (length(gi) >= 3) {
    tabm <- sort(table(colors[gi]), decreasing = TRUE); ifn_mod <- names(tabm)[1]
    ng <- intersect(nfk, names(colors))
    ifn <- data.frame(study = tag, interferon_module = ifn_mod, ifn_genes_in_network = length(gi), ifn_genes_in_that_module = unname(tabm[1]),
                      median_kME_of_those = round(median(kME[gi[colors[gi] == ifn_mod], paste0("ME", ifn_mod)]), 2),
                      nfkb_control_genes_in_network = length(ng), nfkb_control_genes_in_that_module = sum(colors[ng] == ifn_mod),
                      module_size = as.integer(sizes[ifn_mod]), stringsAsFactors = FALSE)
    write.csv(ifn, file.path(out, "ifn_module.csv"), row.names = FALSE)
    cat("\ninterferon module of", tag, ":\n"); print(ifn, row.names = FALSE)
  } else cat("\nFewer than 3 of the 28 interferon genes are in this network: interferon module not identified.\n")

  ## figures: dendrogram with colours (first block) and the module-trait heatmap
  options(repr.plot.width = 11, repr.plot.height = 4.5)
  plotDendroAndColors(net$dendrograms[[1]], colors[net$blockGenes[[1]]], "module", dendroLabels = FALSE, hang = 0.03,
                      addGuide = TRUE, guideHang = 0.05, main = paste(tag, ": gene tree and modules (first block)"))
  dr <- r[, dxcols, drop = FALSE]
  options(repr.plot.width = 10, repr.plot.height = max(5, 0.28 * length(modules) + 3))
  par(mar = c(7, 9, 3, 1))
  labeledHeatmap(Matrix = dr, xLabels = colnames(dr), yLabels = rownames(dr), ySymbols = modules, colorLabels = FALSE,
                 colors = blueWhiteRed(50), textMatrix = signif(dr, 2), setStdMargins = FALSE, cex.text = 0.55, zlim = c(-1, 1),
                 main = paste(tag, ": module eigengene vs diagnosis (correlation)"))
  list(datExpr = datExpr, colors = colors, ME = ME, kME = kME, summary = summ, ifn = ifn, trait = mt, power = POWER, md = md)
}
cat("run_net() defined\n")

---
# Part 5 · RGM network
Samples: all diagnoses and healthy controls, one per patient (more variation than a single disease gives, which helps WGCNA find modules). **Read:** the module list with sizes; the interferon module and whether any NF-kB control gene fell into it; and the module summary, especially any module flagged `batch?`.

In [ ]:
rgm <- run_net("RGM", E_rgm, md_rgm, has_batch = TRUE)
rgm$summary

---
# Part 6 · JDM-lab network
JDM and its healthy controls. The same function, no batch (one lab, one pipeline).

In [ ]:
jdm <- run_net("JDM", E_jdm, md_jdm, has_batch = FALSE)
jdm$summary

---
# Part 7 · Are the modules found in one study present in the other?
**Module preservation** (Langfelder P et al. *PLoS Comput Biol* 2011;7:e1001057), the two **density** statistics Anne uses in notebooks 15 and 26, which need only the study being tested:

| statistic | meaning |
|---|---|
| `mean_cor` | mean pairwise correlation of the module's genes in the other study |
| `prop_var_PC1` | share of the module's variance carried by its first principal component there |

Each is compared with `N_PERM` random gene sets of the same size from the same study, giving a Z score. **Reading** (as Anne's): `none` = the smaller Z is below 2, `weak to moderate` = 2 to 10, `strong` = above 10. Z for `mean_cor` is usually very large because the random sets vary little, so `prop_var_PC1` decides the reading.

This is done **both ways**: RGM modules in the JDM lab, and JDM modules in RGM. A module that is `strong` in both is a gene group that hangs together in two independent labs. Only module gene lists cross between the studies, never expression values.

In [ ]:
stat <- function(Z) { p <- ncol(Z)
  c(mean_cor = (var(rowSums(Z)) - p) / (p * (p - 1)), prop_var_PC1 = svd(Z, nu = 0, nv = 0)$d[1]^2 / sum(svd(Z, nu = 0, nv = 0)$d^2)) }
preserve <- function(mg, target_datExpr, label) {
  Zs <- scale(target_datExpr); Zs <- Zs[, colSums(is.na(Zs)) == 0, drop = FALSE]
  set.seed(SEED)
  res <- do.call(rbind, lapply(split(mg$gene, mg$module), function(g) {
    g_here <- intersect(g, colnames(Zs))
    if (length(g_here) < 10) return(data.frame(genes_in_module = length(g), genes_measured_here = length(g_here),
                                               mean_cor = NA, Z_mean_cor = NA, prop_var_PC1 = NA, Z_prop_var_PC1 = NA))
    obs  <- stat(Zs[, g_here, drop = FALSE])
    null <- replicate(N_PERM, stat(Zs[, sample(colnames(Zs), length(g_here)), drop = FALSE]))
    z    <- (obs - rowMeans(null)) / apply(null, 1, sd)
    data.frame(genes_in_module = length(g), genes_measured_here = length(g_here),
               mean_cor = round(obs[1], 3), Z_mean_cor = round(z[1], 1), prop_var_PC1 = round(obs[2], 3), Z_prop_var_PC1 = round(z[2], 1))
  }))
  res$module <- rownames(res)
  res$reading <- as.character(cut(pmin(res$Z_mean_cor, res$Z_prop_var_PC1), c(-Inf, 2, 10, Inf), labels = c("none", "weak to moderate", "strong")))
  res$reading[is.na(res$reading)] <- "too few genes measured"
  res <- res[order(-res$Z_prop_var_PC1), c("module", "genes_in_module", "genes_measured_here", "mean_cor", "Z_mean_cor", "prop_var_PC1", "Z_prop_var_PC1", "reading")]
  rownames(res) <- NULL; res }

mg_rgm <- read.csv(file.path(WG_DIR, "RGM", "module_genes.csv")); mg_jdm <- read.csv(file.path(WG_DIR, "JDM", "module_genes.csv"))
pres_r_in_j <- preserve(mg_rgm, jdm$datExpr, "RGM modules in JDM lab")
cat("RGM modules, tested in the JDM-lab data:\n"); print(pres_r_in_j, right = FALSE)
write.csv(pres_r_in_j, file.path(WG_DIR, "preservation_RGM_modules_in_JDM.csv"), row.names = FALSE)
pres_j_in_r <- preserve(mg_jdm, rgm$datExpr, "JDM modules in RGM")
cat("\nJDM-lab modules, tested in the RGM data:\n"); print(pres_j_in_r, right = FALSE)
write.csv(pres_j_in_r, file.path(WG_DIR, "preservation_JDM_modules_in_RGM.csv"), row.names = FALSE)

---
# Part 8 · Which module is which? Gene overlap between the two networks
Colour names are arbitrary and differ between studies, so `turquoise` in RGM has nothing to do with `turquoise` in the JDM lab. The overlap table says which RGM module corresponds to which JDM module: for each pair, the genes they share and a hypergeometric p-value (are they sharing more than chance, among the genes present in both networks?). Only pairs sharing at least `MIN_OVERLAP` genes are listed, with the adjusted p. The two **interferon modules** should overlap each other strongly; if they do, the interferon module is the same biology in both labs.

In [ ]:
universe <- intersect(colnames(rgm$datExpr), colnames(jdm$datExpr))
a <- split(intersect(mg_rgm$gene, universe), mg_rgm$module[match(intersect(mg_rgm$gene, universe), mg_rgm$gene)])
b <- split(intersect(mg_jdm$gene, universe), mg_jdm$module[match(intersect(mg_jdm$gene, universe), mg_jdm$gene)])
ov <- do.call(rbind, lapply(names(a), function(x) do.call(rbind, lapply(names(b), function(y) {
  k <- length(intersect(a[[x]], b[[y]]))
  data.frame(RGM_module = x, JDM_module = y, RGM_size = length(a[[x]]), JDM_size = length(b[[y]]), shared = k,
             p = phyper(k - 1, length(a[[x]]), length(universe) - length(a[[x]]), length(b[[y]]), lower.tail = FALSE)) }))))
ov$adj_p <- signif(p.adjust(ov$p, method = "BH"), 3); ov$p <- signif(ov$p, 3)
ov <- ov[ov$shared >= MIN_OVERLAP, ]; ov <- ov[order(ov$adj_p, -ov$shared), ]
ifn_r <- if (!is.null(rgm$ifn)) rgm$ifn$interferon_module else NA; ifn_j <- if (!is.null(jdm$ifn)) jdm$ifn$interferon_module else NA
ov$involves_ifn_module <- ifelse(ov$RGM_module %in% ifn_r | ov$JDM_module %in% ifn_j, "interferon module", "")
rownames(ov) <- NULL
cat("genes present in both networks:", length(universe), "\n")
print(ov, right = FALSE)
write.csv(ov, file.path(WG_DIR, "module_overlap_RGM_JDM.csv"), row.names = FALSE)

---
# Part 9 · Summary and log

In [ ]:
cat("=========================== SUMMARY ===========================\n")
for (nm in c("RGM", "JDM")) {
  x <- get(tolower(nm))
  cat(sprintf("%s: %d samples, %d genes, power %d, %d modules\n", nm, nrow(x$datExpr), ncol(x$datExpr), x$power, ncol(x$ME)))
  if (!is.null(x$ifn)) cat(sprintf("   interferon module: %s (%d of %d interferon genes; %d of %d NF-kB control genes in it)\n",
                                   x$ifn$interferon_module, x$ifn$ifn_genes_in_that_module, x$ifn$ifn_genes_in_network,
                                   x$ifn$nfkb_control_genes_in_that_module, x$ifn$nfkb_control_genes_in_network))
  cat("   modules flagged batch?:", if (any(x$summary$flag == "batch?")) paste(x$summary$module[x$summary$flag == "batch?"], collapse = ", ") else "none", "\n")
}
cat("RGM modules strongly preserved in the JDM lab:", sum(pres_r_in_j$reading == "strong"), "of", nrow(pres_r_in_j), "\n")
cat("JDM modules strongly preserved in RGM:        ", sum(pres_j_in_r$reading == "strong"), "of", nrow(pres_j_in_r), "\n")
if (!is.na(ifn_r) && !is.na(ifn_j)) {
  w <- ov[ov$RGM_module == ifn_r & ov$JDM_module == ifn_j, ]
  cat("the two interferon modules share", if (nrow(w)) w$shared else 0, "genes",
      if (nrow(w)) sprintf("(adjusted p %s)", w$adj_p) else "(fewer than MIN_OVERLAP: not listed)", "\n")
}
log_lines <- c("# WGCNA log (notebook 09)", paste0("Generated ", format(Sys.time(), "%Y-%m-%d %H:%M"), "\n"),
               "## Settings", paste0("MIN_PER_GROUP=", MIN_PER_GROUP, "; RPKM_MIN=", RPKM_MIN, "; N_PERM=", N_PERM, "; seed=", SEED), "",
               "## RGM module summary", "```", capture.output(print(rgm$summary, right = FALSE)), "```", "",
               "## JDM module summary", "```", capture.output(print(jdm$summary, right = FALSE)), "```", "",
               "## RGM modules in JDM", "```", capture.output(print(pres_r_in_j, right = FALSE)), "```", "",
               "## JDM modules in RGM", "```", capture.output(print(pres_j_in_r, right = FALSE)), "```", "",
               "## Module overlap", "```", capture.output(print(ov, right = FALSE)), "```")
writeLines(log_lines, file.path(WG_DIR, "PROCESSING_LOG_wgcna.md"))
print(list.files(WG_DIR, recursive = TRUE))

---
## What to check
1. **Sample trees (Part 5 and 6):** a single sample on a long lone branch is an outlier; tell me if you see one.
2. **Power:** the table shows whether a power reached R-squared 0.90. If none did, the FAQ value was used (Anne's bulk SLE study was the same case).
3. **Interferon module:** most of the 28 genes should be in one module and **none or few** of the 11 NF-kB control genes. If NF-kB genes sit in it, it is a general blood-activation module and not interferon-specific.
4. **Flagged `batch?`:** a module whose strongest link is to the sequencing batch is probably technical. Do not read it as disease.
5. **Preservation:** an RGM module that is `strong` in the JDM lab (and the reverse) is reproduced across labs. `none` means the genes do not hang together in the other study; this is common for small modules.
6. **Overlap table:** the two interferon modules should be each other's best match.

## Limits
- **Few samples for the size of the question.** Anne's own guide (WGCNA FAQ) wants at least 20 samples, ideally more; RGM and the JDM lab have enough, but modules from groups of 10 to 20 samples are less stable.
- **RGM network uses ComBat-seq counts** (diagnosis protected), so some diagnosis structure is built in; modules are descriptive and not a test.
- **The two inputs differ** (counts with PFlog1pPF vs quantile-normalised log RPKM). That is why preservation and overlap use gene lists and correlations, never raw values.
- **Whole blood mixes cell types.** Modules for neutrophil or lymphocyte genes reflect cell mix as much as disease; the hub genes list helps tell which is which.
- **Nothing here is a statistical test of disease.** For tests, see notebooks 05, 07 and 08.

## References
1. Langfelder P, Horvath S. WGCNA: an R package for weighted correlation network analysis. *BMC Bioinformatics* 2008;9:559.
2. Langfelder P, Luo R, Oldham MC, Horvath S. Is my network module preserved and reproducible? *PLoS Comput Biol* 2011;7:e1001057.
3. Booeshaghi AS, Hallgrimsdottir IB, Galvez-Merchan A, Pachter L. Depth normalization for single-cell genomics count data. *bioRxiv* 2022, doi:10.1101/2022.05.06.490859.
4. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (interferon and NF-kB gene sets).
5. Anne Deslattes Mays, endotypes-transcriptomics, notebooks 12 to 15 and 25 to 26.